# Non-negative matrix factorization

The latest version of this code is available on [GitHub](https://github.com/clemens-sirotenko/skip-the-hessian-keep-the-rates).

**Expected runtime:** approximately **1?2 hours** for the default 20-seed comparison with six parallel workers. This estimate is based on recorded CPU runs; hardware and thread settings affect runtime. Reduce the seeds and iteration budgets for a quick check.

Parallel experiments for Section 5.2 and Appendix E.2 of [Skip the Hessian, Keep the Rates](https://arxiv.org/abs/2602.08069) (November 2026 camera-ready version).

Run all cells from a fresh kernel, with the working directory at the repository root or `MF/`. Data and initial factors are generated from the configured seeds; no saved results are required. Each method uses the same data and starting point for each seed.

For factors $U\in\mathbb{R}^{m\times r}$ and $V\in\mathbb{R}^{r\times n}$, the objective is
$$F(U,V)=\tfrac12\|UV-M_{\mathrm{obs}}\|_F^2
+\tfrac\alpha2(\|U\|_F^2+\|V\|_F^2)
+\tfrac\beta2(\|(-U)_+\|_F^2+\|(-V)_+\|_F^2).$$

The cubic Newton baseline requires `galahad-optrove` (see `requirements-glrt.txt`).


In [ ]:
import sys
from pathlib import Path
from datetime import datetime

sys.dont_write_bytecode = True
MF_DIR = Path.cwd()
if not (MF_DIR / "oracles_mat_fac.py").exists():
    MF_DIR = MF_DIR / "MF"
sys.path.insert(0, str(MF_DIR.resolve()))

%reload_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
from utils_new import plot_figures

from joblib import Parallel, delayed, parallel_config, cpu_count
from IPython.display import HTML, display
from time import perf_counter
import importlib.metadata
import platform

%matplotlib inline

# Shared colors for the method comparisons.
method_colors = {
    "Gradient Method": "#E8601C", "PALM": "pink", "LeAP-SSN": "black",
    "GLAd-SSN, p=0.2": "#9B8AC4", "GLAd-SSN, p=0.4": "#88A5DD",
    "GLAd-SSN, p=0.5": "#8DCBE4", "GLAd-SSN, p=0.6": "#C2E3D2",
    "GLAd-SSN, p=0.8": "#DDECBF", "Cubic Newton": "#A97900",
}

## Settings

Configure seeds, dimensions, objective weights, iteration budgets and solver tolerances below. `grad_tol` is the absolute gradient-norm stopping tolerance. Linear systems and cubic subproblems are solved to finite tolerances. `N_JOBS` controls parallel seeds; `THREADS_PER_WORKER` limits numerical-library threads.

Gradient descent runs `4 * gradient_iterations` updates. A PALM iteration consists of both block updates. Set `N_JOBS=1` for sequential execution. For a quick check, use two seeds, `(m,n,r)=(6,4,2)`, and small iteration budgets.


In [ ]:
# Seeds for the 20-seed experiment:
seeds_to_try = [549, 498, 483, 33, 528, 989, 465, 344, 150, 861,
                 107, 424, 998, 235, 992, 229, 725, 911, 418, 146]
#seeds_to_try = [549, 498, 483, 33, 528, 989]

# Data and objective
m, n, r = 200, 100, 12

#m, n, r = 50, 25, 4

noise = 0.2
#alpha, beta = 5.0, 200.0
alpha, beta = 2.0, 200.0

initial_factor_std = 1.0 

# Iteration budgets and regularization
gradient_iterations = 80000
palm_iterations = 150000  # Full U/V sweeps.
number_of_iterations = 4000
grad_tol = 1e-8  # Absolute outer gradient norm; None disables this tolerance.
initial_lambda = 10.0
lss_method = "minres"  # SciPy MINRES; GLRT keeps its compiled GALAHAD backend.
lss_tol = 1e-7
lss_maxiters = 800
gn_exponents = [0.2, 0.5]       # Additional exact/Gauss-Newton comparisons

# Additional nonconvex cubic baseline (independent of initial_lambda)
cubic_initial_H = 10.0
cubic_subproblem_tol = 1e-7  # Model-gradient residual relative to ||g||
cubic_glrt_max_iter = 40000  # Per-trial cap.
cubic_max_backtracks = 200

# Plotting and optional export
plot_threshold = 1e-8
SAVE_FIGURES = True

print("Trying seeds:", seeds_to_try)

# Parallel execution: choose multiple seeds above to use multiple workers.
N_JOBS = 6  # 1 for sequential execution.
THREADS_PER_WORKER = 1


## Run one method for one seed

Generate the seeded problem and invoke the selected solver.


In [ ]:
def run_one_seed(seed, solver_name, options, settings, mf_directory, threads=1):
    """Recreate one seeded problem and run the selected solver."""
    import os
    import sys
    from time import perf_counter
    sys.dont_write_bytecode = True
    if mf_directory not in sys.path:
        sys.path.insert(0, mf_directory)
    import numpy as np
    from threadpoolctl import threadpool_limits, threadpool_info
    import methods
    from oracles_mat_fac import PositiveMatrixFactorization
    from galahad import glrt  # Load native libraries before setting thread limits.

    seed_started = perf_counter()
    with threadpool_limits(limits=threads):
        m, n, r = settings["m"], settings["n"], settings["r"]
        rng = np.random.RandomState(seed)
        U_true = rng.rand(m, r)
        V_true = rng.rand(n, r).T
        M_obs = U_true @ V_true + settings["noise"] * rng.randn(m, n)

        # Preserve the original draw order and the (r, n) layout of V.
        initial_values = settings["initial_factor_std"] * rng.randn(m*r + n*r)
        U_0 = initial_values[:m*r].reshape(m, r)
        V_0 = initial_values[m*r:].reshape(n, r).T
        oracle = PositiveMatrixFactorization(M_obs, r=r, alpha=settings["alpha"], beta=settings["beta"])
        x_0 = oracle.pack(U_0, V_0)

        started = perf_counter()
        _, status, history = getattr(methods, solver_name)(oracle, x_0, **options)
        method_seconds = perf_counter() - started
        cubic = {}
        if solver_name == "cubic_newton_nonconvex_new":
            cubic = {key: np.asarray(history[key]) for key in (
                "hess_vec_calls", "glrt_iterations", "glrt_status",
                "subproblem_relative_residual", "subproblem_calls", "H_k") if key in history}
        return dict(
            seed=int(seed), status=status,
            history={key: np.asarray(history[key]) for key in ("func", "time", "grad_norm", "penalty")},
            cubic=cubic, method_seconds=method_seconds,
            seed_seconds=perf_counter() - seed_started,
            thread_pools=threadpool_info(), pid=os.getpid(),
        )


## Initialize results

Initialize a new experiment with the configured data and seeds. Rerunning this cell clears collected results. Each completed method records its effective options, status and timing.


In [ ]:
experiment_config = dict(
    m=m, n=n, r=r, noise=noise, alpha=alpha, beta=beta,
    initial_factor_std=initial_factor_std,
    gradient_iterations=gradient_iterations, palm_iterations=palm_iterations,
    number_of_iterations=number_of_iterations,
    grad_tol=grad_tol, initial_lambda=initial_lambda,
    lss_method=lss_method, lss_tol=lss_tol, lss_maxiters=lss_maxiters,
    gn_exponents=list(gn_exponents), cubic_initial_H=cubic_initial_H,
    cubic_subproblem_tol=cubic_subproblem_tol, cubic_glrt_max_iter=cubic_glrt_max_iter,
    cubic_max_backtracks=cubic_max_backtracks,
)
experiment_seeds = list(seeds_to_try)
plot_keys = ("func", "time", "grad_norm", "penalty")
effective_workers = min(N_JOBS, len(experiment_seeds), max(1, cpu_count() // THREADS_PER_WORKER))
experiment_threads = THREADS_PER_WORKER

# Shared solver options.
newton_options = dict(
    n_iters=experiment_config["number_of_iterations"], grad_tol=experiment_config["grad_tol"],
    H_0=experiment_config["initial_lambda"], warnings=False,
    lss_tol=experiment_config["lss_tol"], lss_maxiters=experiment_config["lss_maxiters"],
    lss_method=experiment_config["lss_method"], use_gauss_newton=False,
)
methods_to_run = {
    "Gradient Method": ("gradient_method", dict(
        max_iter=experiment_config["gradient_iterations"], grad_tol=experiment_config["grad_tol"], warnings=False)),
    "PALM": ("alternating_gradient_method", dict(
        max_iter=experiment_config["palm_iterations"], tol=experiment_config["grad_tol"],
        L_u0=None, L_v0=None, line_search=True, L_min=1e-12, warnings=False)),
    "LeAP-SSN": ("mf_leapssn", newton_options),
}
for p in (0.2, 0.4, 0.5, 0.6, 0.8):
    methods_to_run[f"GLAd-SSN, p={p:g}"] = ("mf_gladssn", dict(newton_options, p=p))
methods_to_run["Cubic Newton"] = ("cubic_newton_nonconvex_new", dict(
    n_iters=experiment_config["number_of_iterations"], grad_tol=experiment_config["grad_tol"],
    H_0=experiment_config["cubic_initial_H"], subproblem_tol=experiment_config["cubic_subproblem_tol"],
    glrt_max_iter=experiment_config["cubic_glrt_max_iter"],
    max_backtracks=experiment_config["cubic_max_backtracks"], warnings=False,
))
for name in ["LeAP-SSN"] + [f"GLAd-SSN, p={p:g}" for p in experiment_config["gn_exponents"]]:
    solver, options = methods_to_run[name]
    methods_to_run[f"{name} (GN)"] = (solver, dict(options, use_gauss_newton=True))

run_metadata = dict(
    initialized_at=datetime.now().astimezone().isoformat(),
    python=platform.python_version(), platform=platform.platform(), processor=platform.processor(),
    packages={name: importlib.metadata.version(name) for name in
              ("numpy", "scipy", "matplotlib", "joblib", "threadpoolctl", "galahad-optrove")},
    precision="float64", seeds=[int(seed) for seed in seeds_to_try],
    configuration=dict(experiment_config), requested_workers=N_JOBS,
    effective_workers=effective_workers, threads_per_worker=THREADS_PER_WORKER, methods={},
)
all_seeds_histories = {seed: {} for seed in experiment_seeds}
all_seeds_statuses = {seed: {} for seed in experiment_seeds}
all_seeds_timing = {seed: {} for seed in experiment_seeds}
all_seeds_cubic_diagnostics = {}
print(f"Results reset. Using {effective_workers} workers, {experiment_threads} thread(s) each.")


## Reload saved results (optional)

Leave `RESULTS_TO_LOAD=None` for a fresh run. To replot a trusted snapshot generated by this notebook, run imports and settings, load its path below, then continue at **Prepare plots**, skipping initialization and solver execution.


In [ ]:
from experiment_results import load_experiment_snapshot

RESULTS_TO_LOAD = None  # e.g. "parallel_results/raw_20261104_.../results.pkl.gz"
if RESULTS_TO_LOAD is not None:
    results_path = Path(RESULTS_TO_LOAD)
    if not results_path.is_absolute():
        results_path = MF_DIR / results_path
    globals().update(load_experiment_snapshot(results_path))
    print("Restored recorded results from", results_path)
    print("Seeds:", experiment_seeds)
    print("Continue at 'Means over common iterations'; skip initialization and solver cells.")


## Run selected methods

Each method runs all configured seeds and replaces only its own results. Rerun a method after changing its solver controls; changing data or seeds requires reinitialization. Afterward, rerun the plotting section from **Prepare plots**.


In [ ]:
def run_method(name, **option_overrides):
    """Use current solver controls on the fixed experiment; replace one method."""
    import inspect
    import methods

    current_problem = dict(m=m, n=n, r=r, noise=noise, alpha=alpha, beta=beta,
                           initial_factor_std=initial_factor_std)
    changed_problem = [key for key, value in current_problem.items()
                       if value != experiment_config[key]]
    if list(seeds_to_try) != experiment_seeds:
        changed_problem.append("seeds_to_try")
    if changed_problem:
        raise ValueError(
            f"Problem settings changed: {changed_problem}. Restore them to rerun a method "
            "on this experiment, or save your results and initialize a new experiment.")

    # Older snapshots can acquire PALM without clearing/reinitializing other results.
    if name == "PALM" and name not in methods_to_run:
        solver_name, saved_options = "alternating_gradient_method", dict(
            L_u0=None, L_v0=None, line_search=True, L_min=1e-12, warnings=False)
    else:
        solver_name, saved_options = methods_to_run[name]
    allowed_options = set(inspect.signature(getattr(methods, solver_name)).parameters) - {"oracle", "x_0", "z_0"}
    unknown_options = option_overrides.keys() - allowed_options
    if unknown_options:
        raise ValueError(f"Unknown options for {name}: {sorted(unknown_options)}")
    options = dict(saved_options)
    if solver_name == "gradient_method":
        options.update(max_iter=gradient_iterations, grad_tol=grad_tol)
    elif solver_name == "alternating_gradient_method":
        options.update(max_iter=palm_iterations, tol=grad_tol)
    elif solver_name in ("mf_leapssn", "mf_gladssn"):
        options.update(n_iters=number_of_iterations, grad_tol=grad_tol, H_0=initial_lambda,
                       lss_method=lss_method, lss_tol=lss_tol, lss_maxiters=lss_maxiters)
    elif solver_name == "cubic_newton_nonconvex_new":
        options.update(n_iters=number_of_iterations, grad_tol=grad_tol, H_0=cubic_initial_H,
                       subproblem_tol=cubic_subproblem_tol, glrt_max_iter=cubic_glrt_max_iter,
                       max_backtracks=cubic_max_backtracks)
    options.update(option_overrides)  # Explicit per-method choices take precedence.
    if options.get("trace", True) is False:
        raise ValueError("trace=False is not supported: this notebook needs histories for plotting.")
    iteration_key = "max_iter" if solver_name in ("gradient_method", "alternating_gradient_method") else "n_iters"
    budget = options[iteration_key]
    if isinstance(budget, (bool, np.bool_)) or not isinstance(budget, (int, np.integer)) or budget < 0:
        raise ValueError("The iteration budget must be a nonnegative integer.")
    options[iteration_key] = int(budget)
    started_at = datetime.now().astimezone().isoformat()
    started = perf_counter()
    print(f"{name}: running {len(experiment_seeds)} seeds on {effective_workers} workers; "
          f"iteration budget={budget:,}...")
    with parallel_config(backend="loky", inner_max_num_threads=experiment_threads):
        results = Parallel(n_jobs=effective_workers)(
            delayed(run_one_seed)(seed, solver_name, options, experiment_config,
                                  str(MF_DIR.resolve()), experiment_threads)
            for seed in experiment_seeds
        )
    elapsed = perf_counter() - started

    # Store the method only after all its seeds have finished.
    for result in results:
        seed = result["seed"]
        all_seeds_histories[seed][name] = result["history"]
        all_seeds_statuses[seed][name] = result["status"]
        all_seeds_timing[seed][name] = {key: result[key] for key in
            ("method_seconds", "seed_seconds", "thread_pools", "pid")}
        if name == "Cubic Newton":
            all_seeds_cubic_diagnostics[seed] = result["cubic"]
    methods_to_run[name] = (solver_name, dict(options))
    # Keep initialization settings separate from each method's executed options.
    run_metadata["methods"][name] = dict(
        started_at=started_at, elapsed_seconds=elapsed, solver=solver_name, options=dict(options))
    print(f"{name}: all seeds finished in {elapsed:.1f} s.")
    display(pd.DataFrame([
        dict(seed=result["seed"], status=result["status"],
             iterations=len(result["history"]["time"]) - 1,
             gradient=result["history"]["grad_norm"][-1], seconds=result["method_seconds"])
        for result in results
    ]))


### Gradient Method

Uses four times `gradient_iterations`.


In [ ]:
# Reread the current budget without resetting other methods. This reruns from x_0.
run_method("Gradient Method", max_iter=gradient_iterations*4)


### PALM

Alternating gradient updates with blockwise Armijo line search.


In [ ]:
run_method("PALM", max_iter=palm_iterations)
# Optional overrides (edit/uncomment instead of the call above):
# run_method("PALM", max_iter=1500000, tol=None, L_u0=None, L_v0=None, line_search=True)


### LeAP-SSN


In [ ]:
run_method("LeAP-SSN")


### GLAd-SSN, p=0.2


In [ ]:
run_method("GLAd-SSN, p=0.2")


### GLAd-SSN, p=0.4


In [ ]:
run_method("GLAd-SSN, p=0.4")


### GLAd-SSN, p=0.5


In [ ]:
run_method("GLAd-SSN, p=0.5")


### GLAd-SSN, p=0.6


In [ ]:
run_method("GLAd-SSN, p=0.6")


### GLAd-SSN, p=0.8


In [ ]:
run_method("GLAd-SSN, p=0.8")


### Cubic Newton


In [ ]:
run_method("Cubic Newton",glrt_max_iter=cubic_glrt_max_iter)

### LeAP-SSN (GN)


In [ ]:
run_method("LeAP-SSN (GN)")


### GLAd-SSN, p=0.2 (GN)


In [ ]:
run_method("GLAd-SSN, p=0.2 (GN)")


### GLAd-SSN, p=0.5 (GN)


In [ ]:
run_method("GLAd-SSN, p=0.5 (GN)")


## Prepare plots

Collect methods completed for every seed and configure plot selections and axis limits. Objective residuals use the lowest observed objective as their reference.


In [ ]:
completed_methods = [name for name in methods_to_run
                     if all(name in all_seeds_histories[seed] for seed in experiment_seeds)]
assert completed_methods, "Run at least one method cell before plotting."
original_method_names = [name for name in completed_methods if not name.endswith(" (GN)")]
original_method_names = original_method_names or completed_methods

status_table = pd.DataFrame([
    dict(seed=seed, method=name, status=all_seeds_statuses[seed][name],
         gradient=all_seeds_histories[seed][name]["grad_norm"][-1],
         seconds=all_seeds_timing[seed][name]["method_seconds"])
    for seed in experiment_seeds for name in completed_methods
])
display(HTML("<details><summary>Solver results and wall times</summary>" +
             status_table.to_html(index=False, float_format=lambda x: f"{x:.3g}") + "</details>"))

print("Available plot methods:", completed_methods)


## Plot style

Shared colors, markers and legend settings.


In [ ]:
# Shared appearance for representative runs and arithmetic means.
paper_legend_layout = "paper"  # Boxed, translucent legend over the panels.
paper_linewidth = 1.8
paper_marker_spacing = 0.15   # Display spacing; all recorded points are retained.
paper_method_order = ["Gradient Method", "PALM", "Cubic Newton", "LeAP-SSN"]
paper_method_order += [f"GLAd-SSN, p={p:g}" for p in (0.2, 0.4, 0.5, 0.6, 0.8)]
paper_marker_map = dict(zip(paper_method_order, ["None", "None", "*", "s", "^", "D", "x", "v", "h"]))
paper_method_order += [name + " (GN)" for name in paper_method_order
                       if name.startswith(("LeAP", "GLAd"))]
paper_method_colors = dict(method_colors, **{"Cubic Newton": "gold"})


def resolve_plot_options(options, default_methods):
    """Validate one figure's selection without changing any recorded results."""
    requested = options["methods"]
    if requested is not None and not isinstance(requested, (list, tuple)):
        raise ValueError("methods must be a list of method names or None.")
    names = list(default_methods if requested is None else requested)
    if not names or any(not isinstance(name, str) for name in names):
        raise ValueError("Select at least one method by its name.")
    if len(names) != len(set(names)):
        raise ValueError("The methods list contains duplicates.")
    unavailable = [name for name in names if name not in completed_methods]
    if unavailable:
        raise ValueError(f"Methods not recorded for every seed: {unavailable}. "
                         f"Available: {completed_methods}")
    resolved = dict(options, methods=names)
    for key in ("max_time", "max_iter"):
        value = resolved[key]
        if value is not None and (not np.isscalar(value) or not np.isfinite(value) or value <= 0):
            raise ValueError(f"{key} must be a positive number or None.")
    return resolved


## Representative run

Select a shared seed from iteration-to-target comparisons, or supply an explicit seed. Curves show actual recorded values without averaging.


In [ ]:
from representative_plots import select_representative_seed, target_summary

# Selection settings: one method per family; runtime does not affect selection.
representative_selection_methods = [
    "Gradient Method", "LeAP-SSN", "GLAd-SSN, p=0.5", "Cubic Newton",
]
representative_selection_methods = [name for name in representative_selection_methods
                                    if name in completed_methods] or original_method_names
representative_gradient_targets = [1e2, 1.0, 1e-2, 1e-4, 1e-6, 1e-8]
representative_seed_override = None  # An explicit seed is labelled "example".
comparison_gradient_target = 1e-6     # Reporting target, not the solver grad_tol.
representative_zero_scale = 1e-8
representative_penalty_zero_scale = 1e-10  # Linear near zero; keeps zero-to-positive rebounds.

representative_selection = select_representative_seed(
    all_seeds_histories, representative_selection_methods, representative_gradient_targets)
representative_seed = (representative_selection["seed"] if representative_seed_override is None
                       else int(representative_seed_override))
representative_label = "Representative" if representative_seed_override is None else "Example"
print(f"{representative_label} seed: {representative_seed}")
if representative_seed_override is None and len(representative_selection["tied_seeds"]) > 1:
    print("Tied medoids:", representative_selection["tied_seeds"], "-- using the smallest seed.")
if not representative_selection["informative"]:
    print("The milestone ranks do not distinguish these seeds; there is no unique typical run.")
selection_scores = pd.DataFrame([
    {"seed": seed, "medoid score": score, "shown": seed == representative_seed}
    for seed, score in representative_selection["scores"].items()
])
display(selection_scores.style.format({"medoid score": "{:.4f}"}))

In [ ]:
# Edit each figure independently. None selects all completed methods, including GN.
# Explicit lists select only those methods, in the order you specify.
representative_plot_options = {
    "func":      dict(methods=None, max_time=150, max_iter=number_of_iterations),
    "grad_norm": dict(methods=None, max_time=150, max_iter=number_of_iterations),
    "penalty":   dict(methods=None, max_time=150, max_iter=number_of_iterations),
}

# Old paper Figure 4 comparison: uncomment to show its full eight-method set.
representative_plot_options["grad_norm"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]

representative_plot_options["func"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]

representative_plot_options["penalty"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]
# Use "func" or "penalty" instead of "grad_norm" to change another figure.


In [ ]:
# Each figure uses its own methods and zoom; seed selection stays independent.
representative_defaults = [name for name in paper_method_order if name in completed_methods]
representative_options = {
    measure: resolve_plot_options(options, representative_defaults)
    for measure, options in representative_plot_options.items()
}
representative_plot_methods = list(dict.fromkeys(
    name for options in representative_options.values() for name in options["methods"]))
representative_styles = {
    name: dict(color=paper_method_colors[name.removesuffix(" (GN)")],
               linestyle="--" if name.endswith(" (GN)") else "-")
    for name in representative_plot_methods
}
representative_histories = {
    name: all_seeds_histories[representative_seed][name] for name in representative_plot_methods
}
representative_reference = min(
    np.min(representative_histories[name]["func"])
    for name in representative_options["func"]["methods"]
)
representative_figures = {}
representative_plot_records = {}
for measure, reference, ylabel in [
    ("func", representative_reference, "Objective residual"),
    ("grad_norm", 0, "Gradient norm"),
    ("penalty", 0, "Weighted penalty"),
]:
    options = representative_options[measure]
    names = options["methods"]
    scale = (representative_penalty_zero_scale if measure == "penalty"
             and representative_penalty_zero_scale is not None else representative_zero_scale)
    fig = plot_figures(
        [representative_histories[name] for name in names], names,
        colors=[representative_styles[name]["color"] for name in names],
        linewidths=[paper_linewidth] * len(names),
        linestyles=[representative_styles[name]["linestyle"] for name in names],
        n=experiment_config["n"], m=experiment_config["m"], mu_str=None, xlabel="Runtime (s)",
        f_star=reference, threshold=None, zero_scale=scale,
        max_iter=options["max_iter"], max_time=options["max_time"],
        which_measure_1=measure, which_measure_2=measure, ylabel=ylabel,
        suptitle=None, sharey=True, mark_endpoints=False, verbose=False, note=None,
        legend_layout=paper_legend_layout,
        marker_cycle=[paper_marker_map[name.removesuffix(" (GN)")] for name in names],
        markevery=paper_marker_spacing,
    )
    representative_figures[measure] = fig
    representative_plot_records[measure] = dict(
        options, reference=float(reference), zero_scale=scale,
        styles={name: representative_styles[name] for name in names},
        linewidths=[paper_linewidth] * len(names),
    )
    display(fig)
    plt.close(fig)

representative_target_rows = target_summary(
    all_seeds_histories, representative_plot_methods,
    comparison_gradient_target, all_seeds_statuses)
representative_target_table = pd.DataFrame(representative_target_rows)
print(f"Reporting target: {comparison_gradient_target:g}; actual solver grad_tol: "
      f"{run_metadata['configuration']['grad_tol']}")
display(representative_target_table.loc[
    representative_target_table.seed == representative_seed,
    ["method", "reached", "hit_iteration", "hit_time", "final_gradient", "final_penalty", "status"],
].style.format({"hit_iteration": "{:.0f}", "hit_time": "{:.3g}",
                "final_gradient": "{:.3e}", "final_penalty": "{:.3e}"}, na_rep="--"))

# Freeze the settings with the figures; raw histories live in results.pkl.gz.
representative_record = dict(
    format="parallel-nmf-representative-metadata", version=2,
    seed=int(representative_seed), label=representative_label,
    seed_override=representative_seed_override, selection=representative_selection,
    methods=representative_plot_methods, history_keys=list(plot_keys),
    convention="Actual recorded values for one shared seed; no averaging or extension",
    objective_reference=float(representative_reference),
    reporting_gradient_target=comparison_gradient_target,
    plots=representative_plot_records,
    legend_layout=paper_legend_layout, mark_endpoints=False, threshold=None,
    markers=paper_marker_map, marker_spacing=paper_marker_spacing,
    suptitle=None, note=None,
    target_summary=[row for row in representative_target_rows
                    if row["seed"] == representative_seed],
)


## Arithmetic means

Average over all seeds. Completed trajectories retain their final values; the runtime panel uses the latest recorded value at each shared time. Gradient plots apply the configured target and flat-tail display cutoffs.


In [ ]:
# None selects all completed methods, including GN variants, in paper order.
# Each entry controls one two-panel figure; limits only zoom the view.
# Example methods: ["Gradient Method", "LeAP-SSN", "GLAd-SSN, p=0.5"]
mean_plot_options = {
    "func":      dict(methods=None, max_time=180, max_iter=number_of_iterations),
    "grad_norm": dict(methods=None, max_time=180, max_iter=number_of_iterations),
    "penalty":   dict(methods=None, max_time=180, max_iter=number_of_iterations),
}


mean_plot_options["grad_norm"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]

mean_plot_options["func"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]

mean_plot_options["penalty"]["methods"] = [
     "Gradient Method", "Cubic Newton", "LeAP-SSN",
     "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
     "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8",
 ]



In [ ]:
from utils_new import prepare_mean_histories

mean_defaults = [name for name in paper_method_order if name in completed_methods]
mean_options = {
    measure: resolve_plot_options(options, mean_defaults)
    for measure, options in mean_plot_options.items()
}
mean_marker_map = paper_marker_map
mean_marker_spacing = paper_marker_spacing
mean_gradient_target = run_metadata["configuration"]["grad_tol"]  # Display cutoff; solver unchanged.
mean_gradient_flat_tail_rtol = 1e-3  # Hide final plateaus within 0.1%; None disables.

mean_objective_zero_scale = 1e-8  # Objective residual axis only.
mean_penalty_zero_scale = 1e-8  # Penalty axis only; preserve zeros and rebounds.
mean_legend_layout = paper_legend_layout

arithmetic_mean_histories = {}
arithmetic_mean_summaries = {}
for measure in ("func", "grad_norm", "penalty"):
    prepared, summary = prepare_mean_histories(
        all_seeds_histories, mean_options[measure]["methods"], measure=measure,
        target=mean_gradient_target if measure == "grad_norm" else None,
        flat_tail_rtol=mean_gradient_flat_tail_rtol if measure == "grad_norm" else None,
    )
    arithmetic_mean_histories[measure] = prepared
    arithmetic_mean_summaries[measure] = summary

# One observed mean reference, shared by both objective panels and all methods.
mean_objective_reference = min(
    np.min(values) for summary in arithmetic_mean_summaries["func"].values()
    for values in summary["means"].values()
)
arithmetic_mean_figures = {}
mean_plot_records = {}
for measure, reference, ylabel in [
    ("func", mean_objective_reference, "Mean objective residual"),
    ("grad_norm", 0, "Mean gradient norm"),
    ("penalty", 0, "Mean weighted penalty"),
]:
    options = mean_options[measure]
    names = options["methods"]
    prepared = arithmetic_mean_histories[measure]
    summary = arithmetic_mean_summaries[measure]
    limits = {}
    for axis, limit in [("runtime", options["max_time"]), ("iteration", options["max_iter"])]:
        last_shown = max(e["x"] for e in summary[axis]["endpoints"].values())
        limits[axis] = limit if limit is not None else max(last_shown, 1e-12)
    scale = {"func": mean_objective_zero_scale, "grad_norm": 1e-14,
             "penalty": mean_penalty_zero_scale}[measure]
    fig = plot_figures(
        [prepared[name] for name in names], names,
        colors=[paper_method_colors[name.removesuffix(" (GN)")] for name in names],
        linewidths=[paper_linewidth] * len(names),
        linestyles=["--" if name.endswith(" (GN)") else "-" for name in names],
        n=experiment_config["n"], m=experiment_config["m"], mu_str=None, xlabel="Runtime (s)",
        f_star=reference, threshold=None, zero_scale=scale,
        max_time=limits["runtime"], max_iter=limits["iteration"],
        which_measure_1="runtime", which_measure_2="iteration", ylabel=ylabel,
        suptitle=None,  # The paper identifies each figure in its caption.
        sharey=True, runtime_steps=True, verbose=False, note=None,
        legend_layout=mean_legend_layout,
        marker_cycle=[mean_marker_map[name.removesuffix(" (GN)")] for name in names],
        markevery=mean_marker_spacing,
    )
    arithmetic_mean_figures[measure] = fig
    mean_plot_records[measure] = dict(
        options, reference=float(reference), zero_scale=scale, displayed_limits=limits,
        colors={name: paper_method_colors[name.removesuffix(" (GN)")] for name in names},
        linestyles={name: "--" if name.endswith(" (GN)") else "-" for name in names},
        linewidths=[paper_linewidth] * len(names),
    )
    display(fig)
    plt.close(fig)

# Keep the existing gradient results available under their previous names.
mean_gradient_histories = arithmetic_mean_histories["grad_norm"]
mean_gradient_summaries = arithmetic_mean_summaries["grad_norm"]
mean_gradient_figure = arithmetic_mean_figures["grad_norm"]
for axis, summary in mean_gradient_summaries.items():
    missed = [name for name, e in summary["endpoints"].items() if e["all_seeds_reached"] is False]
    trimmed = [name for name, e in summary["endpoints"].items() if e["flat_tail_trimmed"]]
    if missed:
        print(f"{axis}: gradient target missed:", ", ".join(missed))
    if trimmed:
        print(f"{axis}: terminal plateau hidden:", ", ".join(trimmed))

# Keep per-figure method lists/limits with their actual rendered figures.
mean_record = dict(
    format="parallel-nmf-plot-metadata", version=3,
    convention="All-seed arithmetic means with terminal values held; objective/penalty end at last recording, gradient additionally uses target/plateau cutoffs",
    objective_reference=float(mean_objective_reference),
    gradient_target=mean_gradient_target, gradient_flat_tail_rtol=mean_gradient_flat_tail_rtol,
    seeds=list(experiment_seeds), plots=mean_plot_records,
    legend_layout=mean_legend_layout, markers=mean_marker_map, marker_spacing=mean_marker_spacing,
    run_metadata=run_metadata, statuses=all_seeds_statuses,
    summaries={measure: {
        axis: dict(grid_size=len(summary["grid"]),
                   grid_start=float(summary["grid"][0]),
                   grid_end=float(summary["grid"][-1]),
                   endpoints=summary["endpoints"])
        for axis, summary in summaries.items()
    } for measure, summaries in arithmetic_mean_summaries.items()},
)


## Comparison with PALM

Compare gradient norms on one shared seed. `figure8_seed_override=None` uses the representative seed; an explicit value selects another recorded seed. Plot limits affect only the displayed range.


In [ ]:
# First graphic of Figure 8. Edit this list/zoom independently of the other figures.
figure8_plot_options = dict(
    methods=["Gradient Method", "PALM", "LeAP-SSN", "GLAd-SSN, p=0.5"],
    max_time=200, max_iter=3000,
)
figure8_seed_override = None  # None uses representative_seed; otherwise a recorded seed.
figure8_zero_scale = 1e-14    # Preserve exact zeros without cutting off trajectories.


In [ ]:
# Rebuild from recorded histories; clear any earlier figure if methods are missing.
figure8_figure = None
figure8_record = None
figure8_requested = figure8_plot_options["methods"]
figure8_defaults = [name for name in paper_method_order if name in completed_methods]
figure8_missing = [name for name in (figure8_requested or []) if name not in completed_methods]
if figure8_missing:
    print("Figure 8 skipped. Run these methods for all seeds, then rerun Prepare plots and this section:",
          ", ".join(figure8_missing))
else:
    figure8_options = resolve_plot_options(figure8_plot_options, figure8_defaults)
    figure8_names = figure8_options["methods"]
    figure8_seed = representative_seed if figure8_seed_override is None else int(figure8_seed_override)
    if figure8_seed not in all_seeds_histories:
        raise ValueError(f"Seed {figure8_seed} was not recorded. Available seeds: {experiment_seeds}")
    figure8_colors = dict(paper_method_colors, **{"PALM": "pink", "GLAd-SSN, p=0.5": "#9B8AC4"})
    figure8_markers = dict(paper_marker_map, **{"GLAd-SSN, p=0.5": "^"})
    figure8_styles = {
        name: dict(color=figure8_colors[name.removesuffix(" (GN)")],
                   linestyle="--" if name.endswith(" (GN)") else "-",
                   marker=figure8_markers[name.removesuffix(" (GN)")])
        for name in figure8_names
    }
    figure8_figure = plot_figures(
        [all_seeds_histories[figure8_seed][name] for name in figure8_names], figure8_names,
        colors=[figure8_styles[name]["color"] for name in figure8_names],
        linewidths=[paper_linewidth] * len(figure8_names),
        linestyles=[figure8_styles[name]["linestyle"] for name in figure8_names],
        n=experiment_config["n"], m=experiment_config["m"], mu_str=None,
        xlabel="Runtime (s)", ylabel="Gradient norm", f_star=0, threshold=None,
        zero_scale=figure8_zero_scale,
        max_time=figure8_options["max_time"], max_iter=figure8_options["max_iter"],
        which_measure_1="grad_norm", which_measure_2="grad_norm",
        suptitle=None, sharey=True, verbose=False, note=None,
        legend_layout=paper_legend_layout,
        marker_cycle=[figure8_styles[name]["marker"] for name in figure8_names],
        markevery=paper_marker_spacing,
    )
    figure8_record = dict(
        format="parallel-nmf-figure8-metadata", version=1,
        convention="Actual recorded gradient norms for one shared seed; no averaging or extension",
        seed=int(figure8_seed), seed_override=figure8_seed_override,
        representative_seed_override=(representative_seed_override if figure8_seed_override is None else None),
        seed_selection=(representative_selection if figure8_seed_override is None else None),
        **figure8_options, styles=figure8_styles, linewidths=[paper_linewidth] * len(figure8_names),
        legend_layout=paper_legend_layout, marker_spacing=paper_marker_spacing,
        zero_scale=figure8_zero_scale, threshold=None,
        statuses={name: all_seeds_statuses[figure8_seed][name] for name in figure8_names},
    )
    print(f"Figure 8 comparison, seed {figure8_seed}:", ", ".join(figure8_names))
    display(figure8_figure)
    plt.close(figure8_figure)


## Non-negativity penalty

Plot the recorded weighted penalty versus iteration for the representative and PALM comparisons. Method lists, seeds and axis limits can be selected independently; zeros and later increases are retained.


In [ ]:
# Run this options cell, then the plot cell below. No solver rerun is needed.
penalty_iteration_plot_options = {
    "figure3": dict(
        methods=["Gradient Method", "Cubic Newton", "LeAP-SSN",
                 "GLAd-SSN, p=0.2", "GLAd-SSN, p=0.4", "GLAd-SSN, p=0.5",
                 "GLAd-SSN, p=0.6", "GLAd-SSN, p=0.8"],
        max_iter=2000, ylim=None, seed_override=None,
        zero_scale=1e-10, figsize=(6, 4), legend_loc="upper right",
    ),
    "figure8": dict(
        methods=["Gradient Method", "PALM", "LeAP-SSN", "GLAd-SSN, p=0.5"],
        max_iter=2000, ylim=None, seed_override=None,
        zero_scale=1e-10, figsize=(6, 4), legend_loc="upper right",
    ),
}
# Example: zoom only Figure 3 and choose a smaller comparison.
# penalty_iteration_plot_options["figure3"]["max_iter"] = 2000
# penalty_iteration_plot_options["figure3"]["methods"] = ["Gradient Method", "LeAP-SSN", "GLAd-SSN, p=0.5"]
# penalty_iteration_plot_options["figure8"]["ylim"] = (1e-8, 1e5)


In [ ]:
from utils_new import plot_penalty_iterations

# One shared routine, called once for each figure; reset to avoid stale exports.
penalty_iteration_figures = {}
penalty_iteration_records = {}
penalty_defaults = [name for name in paper_method_order if name in completed_methods]
for figure_name, options in penalty_iteration_plot_options.items():
    requested = options["methods"]
    if requested is not None and not isinstance(requested, (list, tuple)):
        raise ValueError(f"{figure_name}: methods must be a list or None.")
    missing = [name for name in (requested or []) if name not in completed_methods]
    if missing:
        print(f"{figure_name} penalty skipped; not recorded for all seeds:", ", ".join(missing))
        continue
    resolved = resolve_plot_options(
        dict(methods=requested, max_iter=options["max_iter"], max_time=None), penalty_defaults)
    names = resolved["methods"]
    inherited_override = (globals().get("figure8_seed_override") if figure_name == "figure8"
                          else None)
    default_seed = representative_seed if inherited_override is None else int(inherited_override)
    seed = default_seed if options["seed_override"] is None else int(options["seed_override"])
    if seed not in all_seeds_histories:
        raise ValueError(f"Seed {seed} was not recorded. Available seeds: {experiment_seeds}")
    colors, markers = dict(paper_method_colors), dict(paper_marker_map)
    if figure_name == "figure8":
        colors["GLAd-SSN, p=0.5"], markers["GLAd-SSN, p=0.5"] = "#9B8AC4", "^"
    styles = {
        name: dict(color=colors[name.removesuffix(" (GN)")],
                   linestyle="--" if name.endswith(" (GN)") else "-",
                   marker=markers[name.removesuffix(" (GN)")])
        for name in names
    }
    fig = plot_penalty_iterations(
        [all_seeds_histories[seed][name] for name in names], names,
        colors=[styles[name]["color"] for name in names],
        linewidths=[paper_linewidth] * len(names),
        linestyles=[styles[name]["linestyle"] for name in names],
        marker_cycle=[styles[name]["marker"] for name in names],
        markevery=paper_marker_spacing,
        max_iter=resolved["max_iter"], ylim=options["ylim"], zero_scale=options["zero_scale"],
        figsize=options["figsize"], legend_loc=options["legend_loc"],
    )
    penalty_iteration_figures[figure_name] = fig
    penalty_iteration_records[figure_name] = dict(
        format="parallel-nmf-penalty-iteration-metadata", version=1,
        figure=figure_name, measure="penalty", seed=int(seed),
        seed_override=options["seed_override"],
        inherited_figure8_seed_override=(inherited_override if options["seed_override"] is None else None),
        representative_seed_override=representative_seed_override,
        methods=names, max_iter=resolved["max_iter"], ylim=options["ylim"],
        zero_scale=options["zero_scale"], figsize=options["figsize"], legend_loc=options["legend_loc"],
        styles=styles, linewidths=[paper_linewidth] * len(names), marker_spacing=paper_marker_spacing,
        convention="Actual weighted penalty at every recorded iteration of one shared seed; zeros and rebounds retained",
        statuses={name: all_seeds_statuses[seed][name] for name in names},
    )
    print(f"{figure_name} penalty, seed {seed}:", ", ".join(names))
    display(fig)
    plt.close(fig)


## Save raw results now

Save histories, settings, seeds, statuses and timings to a new compressed snapshot under `parallel_results/`. Existing snapshots are retained. This does not save solver restart states.


In [ ]:
from experiment_results import save_experiment_snapshot

raw_output_dir = MF_DIR / "parallel_results" / ("raw_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
raw_results_path = save_experiment_snapshot(raw_output_dir, globals())
print("Saved raw results to", raw_results_path)
print("Use this results.pkl.gz path in 'Reload saved results' after restarting.")


## Save figures and data

With `SAVE_FIGURES=True`, export the generated figures, metadata and raw histories into a new timestamped directory under `parallel_results/`.


In [ ]:
if SAVE_FIGURES:
    output_dir = MF_DIR / "parallel_results" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_dir.mkdir(parents=True, exist_ok=False)
    from experiment_results import save_experiment_snapshot
    raw_results_path = save_experiment_snapshot(output_dir, globals())
    print("Saved raw results to", raw_results_path)
    for measure, filename in [("func", "representative_objective.pdf"),
                              ("grad_norm", "representative_gradient.pdf"),
                              ("penalty", "representative_penalty.pdf")]:
        representative_figures[measure].savefig(output_dir / filename, bbox_inches="tight")
    (output_dir / "representative_run.json").write_text(
        json.dumps(dict(representative_record, data_file=raw_results_path.name),
                   indent=2, allow_nan=False), encoding="utf-8")
    mean_output_dir = output_dir
    for measure, filename in [("func", "mean_objective.pdf"), ("grad_norm", "mean_gradient.pdf"),
                              ("penalty", "mean_penalty.pdf")]:
        arithmetic_mean_figures[measure].savefig(mean_output_dir / filename, bbox_inches="tight")
    (mean_output_dir / "arithmetic_mean.json").write_text(
        json.dumps(dict(mean_record, data_file=raw_results_path.name), indent=2, allow_nan=False), encoding="utf-8")
    figure_count = 6
    if globals().get("figure8_figure") is not None:
        figure8_figure.savefig(output_dir / "figure8_gradient.pdf", bbox_inches="tight")
        (output_dir / "figure8_gradient.json").write_text(
            json.dumps(dict(figure8_record, data_file=raw_results_path.name), indent=2, allow_nan=False),
            encoding="utf-8")
        figure_count += 1
        print("Figure 8: figure8_gradient.pdf")
    for figure_name, fig in globals().get("penalty_iteration_figures", {}).items():
        filename = f"{figure_name}_penalty"
        fig.savefig(output_dir / f"{filename}.pdf", bbox_inches="tight")
        (output_dir / f"{filename}.json").write_text(
            json.dumps(dict(penalty_iteration_records[figure_name], data_file=raw_results_path.name),
                       indent=2, allow_nan=False), encoding="utf-8")
        figure_count += 1
        print("Penalty versus iteration:", f"{filename}.pdf")
    print(f"Saved {figure_count} figures, compressed raw results and plot metadata to", output_dir)
    print("Arithmetic means: mean_objective.pdf, mean_gradient.pdf, mean_penalty.pdf")
    print(f"{representative_record['label']} seed {representative_record['seed']}: "
          "representative_objective.pdf, representative_gradient.pdf, representative_penalty.pdf")
else:
    print("Set SAVE_FIGURES=True, then rerun this cell to export the figures and their data (including the Figure 3/8 comparisons when plotted).")
